# ImageForge — actual AI clothing change (Colab prototype)

Real generative editing rather than brush recoloring. SDXL inpainting regenerates only the white part of the mask. After generation the original image is copied unchanged into all black-mask pixels.

**GPU required**: Colab runtime T4 GPU or better. A free GPU is not guaranteed, weights can take several GB, and generation may be slow. This notebook is an experimental test, not an integrated hosted backend.

**Important:** Exact clothing, face identity, pose and prompt adherence cannot be guaranteed. If the red mask preview covers the face or background, fix it before generating.

## Step 1 — GPU + install
Run once. If this reports no CUDA GPU, change Colab runtime to a GPU before continuing.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU. In Colab: Runtime > Change runtime type > T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0), 'VRAM:', round(torch.cuda.get_device_properties(0).total_memory/2**30, 1), 'GiB')
%pip -q install 'diffusers>=0.35,<0.40' 'transformers>=4.45,<5' 'accelerate>=1,<2' 'safetensors>=0.4' 'Pillow>=11,<13'

## Step 2 — Upload the original image (and optionally a mask)

Upload **one original PNG/JPG**. You may also upload a black/white **imageforge-mask.png** exported from ImageForge (where white means edit).

If you don't upload a mask, the notebook uses SegFormer B0 to **suggest clothing areas automatically**. It may not recognize anime clothing correctly. The red-tinted preview shows the selection. If incorrect, do not continue; instead paint the full clothing region in ImageForge, click Save mask, and upload the corrected mask.

In [ ]:
from google.colab import files
from PIL import Image, ImageOps
from IPython.display import display
import io
import numpy as np
uploaded = files.upload()
mask_files = [name for name in uploaded if 'mask' in name.lower()]
image_files = [name for name in uploaded if name not in mask_files]
assert len(image_files) == 1 and len(mask_files) <= 1, 'Upload one image and at most one mask filename containing mask.'
original = ImageOps.exif_transpose(Image.open(io.BytesIO(uploaded[image_files[0]]))).convert('RGB')
assert original.width*original.height <= 16_000_000, 'Image too large (maximum 16 megapixels).'
if mask_files:
    mask = Image.open(io.BytesIO(uploaded[mask_files[0]])).convert('L')
    assert mask.size == original.size, 'Mask must have exactly same width and height as image.'
    mask = mask.point(lambda p: 255 if p >= 128 else 0)
else:
    from transformers import AutoImageProcessor, SegformerForSemanticSegmentation
    model_id='mattmdjaga/segformer_b0_clothes'
    processor = AutoImageProcessor.from_pretrained(model_id)
    segmenter = SegformerForSemanticSegmentation.from_pretrained(model_id).eval().cpu()
    with torch.inference_mode():
        inputs = processor(images=original, return_tensors='pt')
        labels = torch.nn.functional.interpolate(segmenter(**inputs).logits, size=original.size[::-1], mode='bilinear', align_corners=False).argmax(1)[0]
        allowed = torch.zeros_like(labels, dtype=torch.bool)
        for garment_class in [4,5,6,7,8,17]:
            allowed |= (labels == garment_class)
    mask = Image.fromarray((allowed.numpy()*255).astype('uint8'), mode='L')
    del segmenter, processor, inputs, labels, allowed
if mask.getbbox() is None:
    raise ValueError('No clothing detected. Export a manually painted clothing mask from ImageForge.')
print('Edit selection:', round(np.asarray(mask).mean()/255*100, 1), '% of image')
red = Image.blend(original, Image.new('RGB', original.size, (255,25,68)), 0.55)
overlay = Image.composite(red, original, mask)
size = (min(original.width,720), round(original.height*min(1,720/original.width)))
display(original.resize(size))
display(overlay.resize(size))
print('Image 1 is original; image 2 marks the requested edit area in red. STOP if mask is incorrect.')

## Step 3 — Generate the new outfit

Edit the PROMPT to describe the **complete new outfit**. Step 3 downloads SDXL weights on first run and uses GPU. Do not rerun blindly if Colab crashes or runs out of RAM.

This *can* replace clothing detail rather than tint a brush stroke, but it may still change colors inaccurately, produce artifacts, or fail to preserve the original design inside the selected area.

In [ ]:
from diffusers import AutoPipelineForInpainting
PROMPT = 'anime illustration, same character in a blue zip-up hoodie and dark jeans, realistic clothing folds, original art style, consistent scene lighting'
NEGATIVE = 'disfigured anatomy, distorted fabric, blurry, extra limbs, changed face, wrong garment'
STEPS = 25
SEED = 42
scale = min(1.0, 768/max(original.size))
size = (max(256, round(original.width*scale/8)*8), max(256, round(original.height*scale/8)*8))
work_image = original.resize(size, Image.Resampling.LANCZOS)
work_mask = mask.resize(size, Image.Resampling.NEAREST)
pipe = AutoPipelineForInpainting.from_pretrained(
    'diffusers/stable-diffusion-xl-1.0-inpainting-0.1',
    torch_dtype=torch.float16, variant='fp16', use_safetensors=True, low_cpu_mem_usage=True
)
pipe.enable_model_cpu_offload()
pipe.enable_vae_tiling()
pipe.enable_vae_slicing()
generated = pipe(
    prompt=PROMPT, negative_prompt=NEGATIVE, image=work_image, mask_image=work_mask,
    strength=0.98, num_inference_steps=STEPS,
    generator=torch.Generator(device='cpu').manual_seed(SEED)
).images[0].convert('RGB')
edited = Image.composite(generated.resize(original.size, Image.Resampling.LANCZOS), original, mask)
edited.save('imageforge_ai_edit.png')
display(edited.resize(size))
print('Saved: imageforge_ai_edit.png')

## Step 4 — Download edited image

In [ ]:
from google.colab import files
files.download('imageforge_ai_edit.png')